In [25]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

In [26]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [27]:
data = pd.read_csv("/content/drive/MyDrive/dataset/IMDB Dataset.csv")

In [28]:
data.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [29]:
data.shape

(50000, 2)

In [30]:
type(data)

pandas.core.frame.DataFrame

In [31]:
data["sentiment"].value_counts()

,count
sentiment,
positive,25000
negative,25000


# One Hot Encoding
## Label Encoder

In [32]:
data.replace({"sentiment" : {"positive" : 1, "negative" : 0}}, inplace=True)

In [33]:
data.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,1
1,A wonderful little production. <br /><br />The...,1
2,I thought this was a wonderful way to spend ti...,1
3,Basically there's a family where a little boy ...,0
4,"Petter Mattei's ""Love in the Time of Money"" is...",1


# Data Preprocessing

In [34]:
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Embedding, LSTM
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [35]:
train_data, test_data = train_test_split(data, test_size=0.2, random_state=42)

In [36]:
train_data.shape

(40000, 2)

In [37]:
test_data.shape

(10000, 2)

In [38]:
tokenizer = Tokenizer(num_words = 5000)
tokenizer.fit_on_texts(test_data["review"])

In [39]:
X_train = tokenizer.texts_to_sequences(train_data['review'])
X_test = tokenizer.texts_to_sequences(test_data['review'])

In [50]:
X_train = pad_sequences(tokenizer.texts_to_sequences(train_data['review']), maxlen=200)
X_test = pad_sequences(tokenizer.texts_to_sequences(test_data['review']), maxlen=200)

In [51]:
X_train

array([[ 776, 2301,    1, ...,  210,  374, 4301],
       [  14,    3, 1495, ...,   89,  104,    9],
       [   0,    0,    0, ...,    2,  734,   65],
       ...,
       [   0,    0,    0, ..., 1491,    2,  603],
       [   0,    0,    0, ...,  235,  104,  124],
       [   0,    0,    0, ...,   68,   71, 2087]], dtype=int32)

In [52]:
X_test

array([[   0,    0,    0, ..., 1050,  746,  162],
       [  12,  159,   58, ...,  397,    7,    7],
       [   0,    0,    0, ...,   51, 1081,   99],
       ...,
       [   0,    0,    0, ...,  124,  198, 3295],
       [   0,    0,    0, ..., 1080,    1, 2473],
       [   0,    0,    0, ...,    1,  341,   28]], dtype=int32)

In [53]:
Y_train = train_data["sentiment"]
Y_test = test_data["sentiment"]

In [54]:
Y_train

,sentiment
39087,0
30893,0
45278,1
16398,0
13653,0
...,...
11284,1
44732,1
38158,0
860,1


In [55]:
Y_test

,sentiment
33553,1
9427,1
199,0
12447,1
39489,0
...,...
28567,0
25079,1
18707,1
15200,0


In [56]:
model = Sequential()
model.add(Embedding(input_dim=5000, output_dim=128, input_length=200))
model.add(LSTM(128, dropout=0.2, recurrent_dropout=0.2))
model.add(Dense(1, activation='sigmoid'))

In [57]:
model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_2 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_2 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [58]:
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

In [59]:
model.fit(X_train, Y_train, epochs=5, batch_size=64, validation_split=0.2)

Epoch 1/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 219s 420ms/step - accuracy: 0.7244 - loss: 0.5344 - val_accuracy: 0.8602 - val_loss: 0.3364
Epoch 2/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 258s 424ms/step - accuracy: 0.8592 - loss: 0.3437 - val_accuracy: 0.8631 - val_loss: 0.3277
Epoch 3/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 281s 462ms/step - accuracy: 0.8798 - loss: 0.2994 - val_accuracy: 0.8601 - val_loss: 0.3315
Epoch 4/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 212s 425ms/step - accuracy: 0.8880 - loss: 0.2853 - val_accuracy: 0.8765 - val_loss: 0.3075
Epoch 5/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 212s 424ms/step - accuracy: 0.9061 - loss: 0.2386 - val_accuracy: 0.8788 - val_loss: 0.3092


In [61]:
loss, accuracy = model.evaluate(X_test, Y_test)
print("Test Loss:", loss)
print("Test Accuracy:", accuracy)

313/313 ━━━━━━━━━━━━━━━━━━━━ 38s 119ms/step - accuracy: 0.8851 - loss: 0.2999
Test Loss: 0.30151909589767456
Test Accuracy: 0.8831999897956848


In [62]:
def predictive_system(review):
  sequences = tokenizer.texts_to_sequences([review])
  padded_sequences = pad_sequences(sequences, maxlen=200)
  prediction = model.predict(padded_sequences)
  sentiment = "positive" if prediction[0][0] > 0.5 else "negative"
  return sentiment

In [63]:
predictive_system("This movie was fantastic and amazing")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 424ms/step


'positive'

In [64]:
predictive_system("A trilling adventure with stunning visual")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 133ms/step


'positive'

In [65]:
predictive_system("A visual masterpiece")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 134ms/step


'positive'

# Saving Model

In [66]:
model.save("model.h5")

In [67]:
import joblib
joblib.dump(tokenizer, "tokenizer.pkl")

['tokenizer.pkl']